# Bracketed minimization

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/optimization/closed_methods.ipynb)

Like in root finding, you will often have *some idea* of where the optimum is. In similar fashion, we can iteratively improve our brackets until they are *close enough*.

In [ ]:
import numpy as np
import scipy.optimize as optimize

try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

from executable_engineering import ScalarTracker, plot_bracketed_progression

## Golden section search

In the bisection root finding algorithm, two points were sufficient since they identify a sign change. For minimization, we need to determine a minimum, so we need 3 points.

> Given 3 points $(x_1, x_2, x_3)$
>
> Choose a point $x_4$
>
> Evaluate the function at $x_4$
>
> If $f(x_4) < f(x_2)$, select $\langle x_2, x_4, x_3 \rangle$.
>
> If $f(x_4) > f(x_2)$, select $\langle x_1, x_2, x_4 \rangle$.
>
> Repeat until a tolerance is met

How do we choose $x_4$?

![Golden Section Search](images/golden_section_search.png)

### The Golden ratio

To make a robust method, we want both of these paths to result in the same reduction of the search interval, and ideally reuse the function evaluations. This *self-similarity* implies we want to maintain the ratio of subdivision:

In the triplet $\langle x_1, x_2, x_4 \rangle$ with $f_{4a}$, we want $\frac{c}{a}=\frac{a}{b}$.

In the triplet $\langle x_2, x_4, x_3 \rangle$ with $f_{4b}$, we want $\frac{c}{b-c}=\frac{a}{b}$.

Using these two equations, we can eliminate $c$ and find,

$$
\left(\frac{b}{a}\right)^2 - \frac{b}{a} = 1
$$

and ignoring the negative root,

$$
\frac{b}{a} = \varphi = \frac{1+\sqrt{5}}{2} = 1.618033988\dots
$$

the *Golden ratio*!

## Convergence rate

Each iteration of the Golden section method reduces the bracketed interval by a constant factor of $\varphi^{-1} \approx 0.618$.

The Golden section method is guaranteed to converge, doing so reliably at a linear rate ($k=1$ with rate constant $\varphi^{-1} \approx 0.618$).

Let's see this on the unimodal function $f(x) = x^4 + 10x$ (where the true minimum is at $x^* = -(10/4)^{1/3} \approx -1.3572$):

In [ ]:
def f(x):
    return x**4 + 10 * x

x_true = -(10 / 4)**(1 / 3)
tracker = ScalarTracker(f)
result = optimize.minimize_scalar(tracker, bracket=(-5, 0, 5), method='Golden')
print("Minimum found at x =", result.x)
print("Minimum function value =", result.fun)

plot_bracketed_progression(f, tracker, result, x_range=(-5, 5), title="Golden Section Search on f(x) = x^4 + 10x")

## Brent's method

We've seen this concept before in root finding with bisection. Later we saw the value of using the function values to approximate the derivative and guess at the root. Since we had two points, we fit a line (which approximated the derivative) which is the Secant method.

Now we have 3 points so we can fit a parabola and find $f_{\min} = f(x_{\min})$!

### Parabolic interpolation

Fitting a parabola through the three points $(x_1, f_1)$, $(x_2, f_2)$, and $(x_3, f_3)$ allows us to jump directly to the vertex of that parabola as our next estimate $x_4$:

$$
x_4 = x_2 - \frac{1}{2} \frac{(x_2-x_1)^2(f_2-f_3) - (x_2-x_3)^2(f_2-f_1)}{(x_2-x_1)(f_2-f_3) - (x_2-x_3)(f_2-f_1)}
$$

(Note: For *root finding*, where we want $x$ where $f(x)=0$, Brent's root-finding algorithm uses *inverse quadratic interpolation* to fit $x(f)$. For *minimization*, where we want $f'(x)=0$, Brent's minimization method uses *successive parabolic interpolation* as written above!)

The convergence order of parabolic interpolation on smooth functions is superlinear ($k \approx 1.325$).

### Combination with Golden section

Brent's method combines the robustness of the Golden Section method with the acceleration of parabolic approximation (still without any derivative information!).

Candidate steps from parabolic interpolation and Golden section are generated and compared against various criteria for robustness and progression.

This decision tree makes the exact convergence behavior adaptive, achieving superlinear convergence ($k \approx 1.325$) for well-behaved functions while guaranteeing at least the linear convergence rate of Golden section search.

## Example: Unimodal minimization

Minimize $f(x) = x^4 + 10x$ using Brent's method and compare the evaluation count and convergence curve against Golden section search:

In [ ]:
tracker = ScalarTracker(f)
result = optimize.minimize_scalar(tracker, bracket=(-5, 0, 5), method='Brent')
print("Minimum found at x =", result.x)
print("Minimum function value =", result.fun)

plot_bracketed_progression(f, tracker, result, x_range=(-5, 5), title="Brent's Method on f(x) = x^4 + 10x")

## Example: Non-differentiable function

Let's try a non-differentiable function: $f(x) = |x|$:

In [ ]:
def nondiff_func(x):
    return np.abs(x)

# What should we use for the bracket? What happens if we use a different method?
bracket = (-5, 1, 5)

tracker = ScalarTracker(nondiff_func)
result = optimize.minimize_scalar(tracker, bracket=bracket, method='Brent')
print("Minimum found at x =", result.x)
print("Minimum function value =", result.fun)

plot_bracketed_progression(nondiff_func, tracker, result, x_range=(-5, 5), title="Brent's Method on f(x) = |x|")

## Example: A more treacherous function

Minimize $f(x) = |\sin(x)| + |\cos(x)| + 0.1 x^2$

In [ ]:
def complicated_nondiff_func(x):
    return np.abs(np.sin(x)) + np.abs(np.cos(x)) + 0.1 * x**2

tracker = ScalarTracker(complicated_nondiff_func)
result = optimize.minimize_scalar(tracker, bracket=(-5, 1, 5), method='Brent')
print("Minimum found at x =", result.x)
print("Minimum function value =", result.fun)

plot_bracketed_progression(complicated_nondiff_func, tracker, result, x_range=(-5, 5), title="Brent's Method on a Complicated Non-Differentiable Function")